In [1]:
# Importation des librairies
import pandas as pd
import numpy as np
import os

In [2]:
# Importer la dataframe
df = pd.read_csv(os.path.join(os.getcwd(), '..', 'data', 'predictive_maintenance.csv'))

# 1.) Suppression de variables

*Ces sous-types de panne (TWF, HDF, PWF, OSF, RNF) sont en réalité les causes de Machine failure . Si nous les utilisez comme features pour prédire Machine failure, nous risquez du data leakage massif puisqu'ils encodent directement la cible. Nous allons les exclure du modèles.*

In [3]:
# Suppression des variables de cause de panne (pour éviter le data leakage)
df = df.drop(columns=['UDI','Product ID'], errors='ignore')

# 2.) Création de nouvelles variables

Nous allons créer de nouvelles variables.

In [4]:
# Feature 1 : Différentiel thermique (engineering insight)
df['temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']

In [5]:
# Feature 2 : Puissance mécanique estimée (P = T × ω)
df['power_kw'] = (df['Torque [Nm]'] * df['Rotational speed [rpm]'] * 2 * np.pi / 60) / 1000

In [6]:
# Feature 3 : Indice de contrainte thermique
df['thermal_stress'] = df['temp_diff'] * df['power_kw']
df.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,temp_diff,power_kw,thermal_stress
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,10.5,6.951591,72.991701
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,10.5,6.826723,71.680589
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,10.4,7.749388,80.593630
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,10.4,5.927505,61.646048
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,10.5,5.897817,61.927074


# 3.) Normalisation des données numériques

Comme nous avons gardez les outliers dans le jeu de données, RobustScaler (basé sur médiane et IQR) est le meilleur choix.

Cependant, il y a un risque de data leakage léger (les statistiques médiane/IQR seront calculées sur train+test combinés). Pour rigueur méthodologique complète, il serait préférable de faire le split avant le scaling, puis de fit le scaler uniquement sur X_train

In [7]:
# from sklearn.preprocessing import RobustScaler

# # Colonnes numériques à normaliser (hors identifiants, cibles et variables binaires)
# numeric_cols = ['Air temperature [K]', 'Process temperature [K]', 
#                  'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
#                  'temp_diff', 'power_kw', 'thermal_stress']

# scaler = RobustScaler()
# df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# df.head()

# 4.) Encodage des données catégorielles

In [8]:
# Label/Ordinal Encoding (il y a un ordre L < M < H)
type_mapping = {'L': 0, 'M': 1, 'H': 2}
df['Type_encoded'] = df['Type'].map(type_mapping)
df = df.drop(columns=['Type'])
df.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,temp_diff,power_kw,thermal_stress,Type_encoded
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,10.5,6.951591,72.991701,1
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,10.5,6.826723,71.680589,0
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,10.4,7.749388,80.593630,0
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,10.4,5.927505,61.646048,0
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,10.5,5.897817,61.927074,0


# Exporter la dataframe

In [9]:
# Export du dataframe transformé au format CSV dans le même dossier que predictive_maintenance.csv
df.to_csv(os.path.join(os.getcwd(), '..', 'data', 'predictive_maintenance_prepa.csv'), index=False)